In [3]:
import pandas as pd
import numpy as np
import random

random.seed(42)
np.random.seed(42)

n = 800  # plus de cas
data = []

for _ in range(n):
    
    risque = random.choices(
        ['Normale', 'Modere', 'Eleve'],
        weights=[60, 25, 15]
    )[0]

    if risque == 'Normale':
        bpm          = random.randint(58, 115)  # large chevauchement
        temperature  = round(random.uniform(36.0, 38.2), 1)
        spo2         = random.randint(93, 100)
        tension_s    = random.randint(95, 142)  # large chevauchement
        tension_d    = random.randint(60, 92)
        contractions = random.randint(0, 3)
        semaine      = random.randint(8, 40)

    elif risque == 'Modere':
        bpm          = random.randint(85, 135)  # chevauchement fort
        temperature  = round(random.uniform(37.0, 39.2), 1)
        spo2         = random.randint(90, 97)
        tension_s    = random.randint(118, 158)
        tension_d    = random.randint(78, 100)
        contractions = random.randint(1, 6)
        semaine      = random.randint(8, 40)

    else:  # Eleve
        bpm          = random.randint(105, 160)
        temperature  = round(random.uniform(37.8, 41.0), 1)
        spo2         = random.randint(82, 95)  # chevauchement
        tension_s    = random.randint(130, 185)
        tension_d    = random.randint(85, 125)
        contractions = random.randint(3, 12)
        semaine      = random.randint(8, 40)

    # Bruit réaliste sur TOUS les paramètres — 20% des cas
    if random.random() < 0.20:
        bpm          = bpm + random.randint(-20, 20)
        temperature  = round(temperature + random.uniform(-0.5, 0.5), 1)
        spo2         = spo2 + random.randint(-3, 3)
        tension_s    = tension_s + random.randint(-15, 15)
        tension_d    = tension_d + random.randint(-10, 10)

        # Garder dans des limites physiologiques
        bpm          = max(40, min(180, bpm))
        temperature  = max(34.0, min(42.0, temperature))
        spo2         = max(70, min(100, spo2))
        tension_s    = max(60, min(200, tension_s))
        tension_d    = max(40, min(130, tension_d))

    # Cas paradoxaux — une femme Normale peut avoir
    # un paramètre légèrement anormal (stress, effort)
    if risque == 'Normale' and random.random() < 0.08:
        bpm = random.randint(105, 120)  # tachycardie de stress

    if risque == 'Normale' and random.random() < 0.05:
        tension_s = random.randint(135, 145)  # tension blouse blanche

    data.append({
        'bpm'                    : bpm,
        'temperature'            : temperature,
        'spo2'                   : spo2,
        'tension_systolique'     : tension_s,
        'tension_diastolique'    : tension_d,
        'contractions_par_10min' : contractions,
        'semaine_grossesse'      : semaine,
        'risque'                 : risque
    })

df = pd.DataFrame(data)

# Feature Engineering
df['pulse_pressure'] = (
    df['tension_systolique'] - df['tension_diastolique']
)
df['indice_tension'] = (
    df['tension_systolique'] / df['tension_diastolique']
).round(2)
df['fievre']     = (df['temperature'] >= 37.6).astype(int)
df['spo2_basse'] = (df['spo2'] < 95).astype(int)

def get_trimestre(s):
    if s <= 12: return 1
    elif s <= 26: return 2
    else: return 3

df['trimestre'] = df['semaine_grossesse'].apply(get_trimestre)

df.to_csv('dataset_mamaguard2.csv', index=False)

print("=" * 45)
print("   DATASET MAMAGUARD V3")
print("=" * 45)
print(f"\n Cas total       : {len(df)}")
print(f" Colonnes        : {len(df.columns)}")
print(f"\n Repartition :")
print(df['risque'].value_counts())

# Vérifier le chevauchement entre classes
print(f"\n Verification du chevauchement BPM :")
for r in ['Normale', 'Modere', 'Eleve']:
    sub = df[df['risque'] == r]['bpm']
    print(f"   {r:<10} → min={sub.min()} max={sub.max()} moy={sub.mean():.1f}")

print(f"\n Verification tension systolique :")
for r in ['Normale', 'Modere', 'Eleve']:
    sub = df[df['risque'] == r]['tension_systolique']
    print(f"   {r:<10} → min={sub.min()} max={sub.max()} moy={sub.mean():.1f}")

   DATASET MAMAGUARD V3

 Cas total       : 800
 Colonnes        : 13

 Repartition :
risque
Normale    490
Modere     215
Eleve       95
Name: count, dtype: int64

 Verification du chevauchement BPM :
   Normale    → min=40 max=129 moy=88.9
   Modere     → min=78 max=150 moy=110.3
   Eleve      → min=101 max=176 moy=131.9

 Verification tension systolique :
   Normale    → min=82 max=151 moy=119.3
   Modere     → min=111 max=170 moy=139.7
   Eleve      → min=125 max=193 moy=157.8
